In [ ]:
"""

This is just a barebones implementation that I will be able to build off of in the future.
In the future I plan to add:
    Compatibility with multiple architectures (ResNet-18, ResNet-52, ..., LetNet-5?, AlexNNet?, ...)
    All of the metrics from the paper
    Different types of batchselection methods (DivBS, Full?, RHO-Loss?, ...)
    ...

Steps:

1. Load libraries & initialize values

2. Load CIFAR10 dataset and split into training and validation sets

3. Initialize ResNet-18 model with random weights

4. Train model using uniform batch selection
    log and save both training and validation accuracy at each epoch

5. Plot metrics (epoch vs. training and validation accuracy)

"""

'\n\nThis is just a barebones implementation that I will be able to build off of in the future.\nIn the future I plan to add:\n    Compatibility with multiple architectures (ResNet-18, ResNet-52, ..., LetNet-5?, AlexNNet?, ...)\n    All of the metrics from the paper\n    Different types of batchselection methods (DivBS, Full?, RHO-Loss?, ...)\n    ...\n\nSteps:\n\n1. Load libraries & initialize values\n\n2. Load CIFAR10 dataset and split into training and validation sets\n\n3. Initialize ResNet-18 model with random weights\n\n4. Train model using uniform batch selection\n    log and save both training and validation accuracy at each epoch\n\n5. Plot metrics (epoch vs. training and validation accuracy)\n\n'

In [ ]:

# 1. Load libraries & initialize values


In [ ]:
!pip install wandb -qU

In [ ]:
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split
import random

In [ ]:
import wandb
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [ ]:
def random_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
run = wandb.init(entity="connerdraper-byu",
                 project="DivBS",
                 config={
                     "epochs": 25,
                     "batch_size": 64,
                     "lr": 1e-3,
                     "budget": .2,
                     "seed": 27,
                     "device": "cuda" if torch.cuda.is_available() else "cpu",
                     "dataset": "CIFAR10",
                     "architecture": "ResNet-18",
                     "loss_function": "CrossEntropyLoss",
                     "optimizer": "Adam",
                     "metrics": ["train_acc", "val_acc"],
                 }
)

In [ ]:

# 2. Load CIFAR10 dataset and split into training, validation, and testing data


In [ ]:
# below is just from Gemini
# but we'll need to modify it to have a custom dataloader, a dataloader that prunes our batches
# our simple implementation will just be uniform random selection but is an important, basic implementation that we will later modify (as we add DivBS)

In [ ]:
# create transform object (for normalization)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# get train, validation, and testing CIFAR10 datasets
full_train = CIFAR10(root='./data', train=True, download=True, transform=transform)
train_dataset, val_dataset = random_split(full_train, [45000, 5000])
test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)

# create train/test DataLoader objects
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# important functions & values

def uniform_batch_selection(images, labels, budget=0.2):
    n = len(images)
    n_selected = max(2, int(n * budget))
    indices_selected = np.random.choice(n, size=n_selected, replace=False)
    return images[indices_selected], labels[indices_selected]

def full_batch_selection(images, labels):
    return images, labels

In [ ]:
# this uses algorithm 1 from the paper
# it didn't get great results. I think algorithm 2 would be much better
# I think the paper's results are from algo 2 and i was jsut dumb oops
def diverse_batch_selection(images, labels, model, budget=0.2):
    model.eval()

    n = len(images)
    n_selected = max(2, int(n * budget))

    images.requires_grad = True
    outputs = model(images)
    loss = torch.nn.functional.cross_entropy(outputs, labels)
    grad_out = torch.autograd.grad(loss, outputs, retain_graph=True)[0]

    S, E = [], []
    remaining = list(range(n))
    Sum = grad_out.sum(dim=0)

    while len(S) < n_selected:
        candidates = []

        for i in remaining:
            gi = grad_out[i]
            if E:
                proj = sum(torch.dot(e, gi) * e for e in E)
                ei = gi - proj
            else:
                ei = gi
            ei = ei / (ei.norm() + 1e-12)
            candidates.append(ei)

        E_candidate = torch.stack(candidates)
        scores = torch.abs(E_candidate @ Sum)
        best_idx_local = torch.argmax(scores).item()
        best_idx = remaining[best_idx_local]

        S.append(best_idx)
        E.append(E_candidate[best_idx_local])
        remaining.remove(best_idx)

    S_tensor = torch.tensor(S, device=images.device)
    return images[S_tensor], labels[S_tensor]


In [ ]:

# 3. Initialize ResNet-18 model with random weights


In [ ]:
from torchvision.models import resnet18
import torch.nn as nn

model = resnet18(weights=None)  # keeps conv1 expecting 3 channels
model.fc = nn.Linear(512, 10)   # adjust output for CIFAR-10
model = model.to(wandb.config.device)

In [ ]:
wandb.watch(model, log="all", log_freq=100)

In [ ]:
# create optimizer and loss objects
optimizer = optim.Adam(model.parameters(), lr=wandb.config.lr)
criterion = nn.CrossEntropyLoss()

In [ ]:

# 4. Train model using uniform batch selection
#     log and save both training and validation accuracy at each epoch


In [ ]:
print(f"we run on: {wandb.config.epochs} epochs")

we run on: 25 epochs


In [ ]:
# 2. Watch model
wandb.watch(model, log="all", log_freq=1000)

global_step = 0
for epoch in range(wandb.config.epochs):
    # Training loop
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(wandb.config.device), labels.to(wandb.config.device)
        sel_imgs, sel_labels = diverse_batch_selection(images, labels, model, budget=wandb.config.budget)

        outputs = model(sel_imgs)
        loss = criterion(outputs, sel_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        acc = (outputs.argmax(1) == sel_labels).float().mean().item()
        # log per-batch metrics
        wandb.log({"train/loss": loss.item(), "train/acc": acc})
        global_step += 1

    # Validation at epoch end
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(wandb.config.device), labels.to(wandb.config.device)
            out = model(images)
            val_loss += criterion(out, labels).item()
            val_correct += (out.argmax(1) == labels).sum().item()
            val_total += labels.size(0)
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    wandb.log({"val/loss": val_loss, "val/acc": val_acc})

wandb: WARNING Tried to log to step 0 that is less than the current step 1171. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 1 that is less than the current step 1875. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 2 that is less than the current step 2579. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 3 that is less than the current step 3283. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING Tried to log to step 4 that is less than the current step 3987. Steps must be monotonically increasing, so this data will be ignored. See https:/

In [ ]:
# 3. Log gradient histograms
for name, param in model.named_parameters():
    wandb.log({f"grads/{name}": wandb.Histogram(param.grad.cpu())}, step=global_step)

In [ ]:
# 4. Log sample predictions
imgs, labels = next(iter(val_loader))
preds = model(imgs.to(wandb.config.device))
examples = [
    wandb.Image(img, caption=f"pred={p}, label={l}")
    for img, p, l in zip(imgs, preds.argmax(1), labels)
]
wandb.log({"examples": examples})

In [ ]:
# 5. Save model artifact
artifact = wandb.Artifact("divbs-model", type="model")
artifact.add_file("model.ckpt")
wandb.log_artifact(artifact)

ValueError: Path is not a file: 'model.ckpt'